# Scherrer, Lynch et al. (2023), dandiset 000691: decoding octant

Paper. Scherrer JR, Lynch GF, Zhang JJ, Fee MS (2023). An optical design enabling lightweight
and large field-of-view head-mounted microscopes. *Nature Methods* 20, 546-549.
[doi:10.1038/s41592-023-01806-1](https://doi.org/10.1038/s41592-023-01806-1).

Claims. `claims/000691/10.1038-s41592-023-01806-1.yaml`, written from the full text of the
version of record and the paper's code (GitHub `FeeLab/Feescope-Paper-Code`, commit d596646). It
decides what is tested and how.

Dandiset. [DANDI:000691](https://dandiarchive.org/dandiset/000691/0.260827.1358), version
0.260827.1358, the only published version (the paper cites none); see the main notebook,
`reproduce_scherrer_2023.ipynb`, for the reasons and the full data check. This notebook holds the
octant decoding claims, apart to keep each notebook within the time limit.

Footprints. The claims use the 2,382 curated footprints, with all 2,948 as a named variant.
The dandiset does not mark the curated set, so all 2,948 are used.

| Claim | Data | Status |
|---|---|---|
| ed9b-octant-decoding | dandiset | run under null (n1); (n2) not run |
| ed9c-local-decoding-map | dandiset | not run; cost measured |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import kiloscope as ks

pd.set_option("display.width", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)
started = time.time()
urls = ks.asset_urls()
f = ks.open_nwb(urls)
S = ks.load_session(f)
POS = ks.mask_positions(ks.read_masks(urls[ks.NWB_PATH], f))
track, B = ks.load_behaviour(urls, S)
T = S["traces"]
run = B["running"]
X_run = ks.moving_max(T[B["sel"]])[run]
print(f"loaded and tracked in {time.time() - started:.0f} s")

## Data check

The full check of the data is in the main notebook. Here: what this notebook decodes.

In [ ]:
LABELS = B["octant"][run]
LAMBDA = ks.LAMBDA_OTHER
NAMES = [f"octant {k + 1}" for k in range(8)]
print(f"imaging frames within the tracked video: {len(B['sel'])}; running (speed above its median): {run.sum()}")
print(f"activity: moving maximum over 30 frames of {X_run.shape[1]} footprints; values {X_run.min():.3f} to {X_run.max():.3f}")
print("frames per class:", dict(zip(NAMES, np.bincount(LABELS, minlength=8).tolist())))
n = len(LABELS)
print(f"each fit trains on {round(n * ks.HOLDOUT) - round(n * ks.BUFFER)} frames and tests on {n - round(n * ks.HOLDOUT) - round(n * ks.BUFFER)}; "
      f"lambda {LAMBDA}; solver for {X_run.shape[1]} predictors: {ks._solver(X_run, None)}")

The labels come from the position tracked in the behaviour video with the code's rule (checked in
the main notebook). Every class occurs in the running frames (printed above).

## ed9b-octant-decoding

> Position within the maze can be decoded with octant resolution from the activity, above
> decoders trained on shuffled data.

Criterion: eight octants (direction not coded), lambda 6.75e-8, the average matrix of 200 fits;
the null is 1,000 decoders on their own shuffles, under the readings (n1) one fit per shuffle and
(n2) the average of 200 fits per shuffle. Reproduced when the real median accuracy exceeds every
null value (p < 0.001) and exceeds chance (1/8) under both; not reproduced when it fails under
both; otherwise methods_underspecified.

In [ ]:
rng_real = np.random.default_rng(69130)
t0 = time.time()
real = ks.decoding_matrix(X_run, LABELS, 8, LAMBDA, 200, rng_real, n_jobs=4)
sec_real = (time.time() - t0) / 200
print(f"200 fits in {time.time() - t0:.0f} s ({sec_real:.2f} s per fit, 4 in parallel); classes absent from a test part: {real['absent']} of 1600")
print(f"median accuracy (median of the diagonal of the average matrix): {real['median']:.3f}; chance 0.125; ratio {real['median'] / 0.125:.2f}")
print(pd.DataFrame({"class": NAMES, "accuracy": real["diag"].round(3),
                    "per_fit_5th": np.nanpercentile(real["fit_diag"], 5, axis=0).round(3),
                    "per_fit_95th": np.nanpercentile(real["fit_diag"], 95, axis=0).round(3)}).to_string(index=False))
t0 = time.time()
X_F = np.asfortranarray(X_run)         # columns contiguous, for fast shuffles
null1 = ks.null_medians(X_F, LABELS, 8, LAMBDA, 1000, 1, seed=69131, n_jobs=4)
sec_null = (time.time() - t0) / 1000
print(f"\nnull (n1): 1000 shuffles, one fit each, in {time.time() - t0:.0f} s; NaN medians {np.isnan(null1).sum()}")
print(f"  null median accuracies: percentiles 0/50/95/99/100 {np.round(np.nanpercentile(null1, [0, 50, 95, 99, 100]), 3)}")
k = int((null1 >= real["median"]).sum())
print(f"  null values at or above the real {real['median']:.3f}: {k}; p = {k / 1000:.4f} (fraction), {(k + 1) / 1001:.4f} ((k + 1) / 1001)")
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
im = ax[0].imshow(real["matrix"], cmap="RdBu_r", vmin=0, vmax=2 * 0.125 + max(0, np.nanmax(real["matrix"]) - 0.25))
ax[0].set_xticks(range(8), NAMES, rotation=45, fontsize=7); ax[0].set_yticks(range(8), NAMES, fontsize=7)
ax[0].set_xlabel("actual"); ax[0].set_ylabel("predicted"); plt.colorbar(im, ax=ax[0])
ax[1].hist(null1, 40, color="0.6", label="null (n1)"); ax[1].axvline(real["median"], color="r", label="decoders")
ax[1].axvline(0.125, color="k", ls=":", label="chance"); ax[1].set_xlabel("median accuracy"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()
print(f"matrix values {np.nanmin(real['matrix']):.3f} to {np.nanmax(real['matrix']):.3f}; null values {np.nanmin(null1):.3f} to {np.nanmax(null1):.3f}, all inside the axes")

In [ ]:
# Null (n2): each null decoder is the average matrix of 200 fits on its shuffle. One such decoder is run to show
# where these values lie; the 1,000 the criterion asks for are not run (cost below).
t0 = time.time()
null2_sample = []
for s_ in np.random.SeedSequence(69132).spawn(1):
    rng_s = np.random.default_rng(s_)
    null2_sample.append(ks.decoding_matrix(ks.shuffle_columns(X_F, rng_s), LABELS, 8, LAMBDA, 200, rng_s, n_jobs=4)["median"])
print(f"one (n2) null decoder, 200 fits, in {time.time() - t0:.0f} s: median accuracy {np.round(null2_sample, 3).tolist()} "
      f"(the real decoder {real['median']:.3f})")
fits_n2 = 1000 * 200
print(f"(n2) needs {fits_n2} fits; at the {sec_null:.2f} s per fit measured above with 4 fits in parallel, {fits_n2 * sec_null / 3600:.1f} hours "
      f"on this machine, where a notebook may take 30 minutes")

**Result: blocked (not_runnable).** Under null reading (n1) the claim holds: the average matrix of
200 fits has a median accuracy of 0.326, 2.61 times chance; none of the 1000 shuffled decoders
reaches it (the largest is 0.229), so p < 0.001, and it exceeds chance. Reading (n2), 1000 shuffles
of 200 fits each, needs 200,000 fits, about 38 hours on this machine; the one (n2) null decoder run
gives 0.110. The criterion asks for both readings, so the claim cannot be decided within the time
limit.

## ed9c-local-decoding-map

> Decoders built from the 32 footprints closest to a point decode octant position significantly in parts of the field of view, with accuracy that varies across it.

Criterion: map points every 5 image pixels over rows 1 to 608 and columns 140 to 700; a point is
used when at least 32 footprints lie within 270 um (45 pixels at 5.94 um, or 50 at 5.35); at each,
the average matrix of 40 fits on its 32 nearest footprints; the null is 1,000
decoders on shuffled data from 32 random footprints (one fit or 40 each);
Benjamini-Hochberg 5%. It depends on ed9b-octant-decoding. Its cost is measured here.

In [ ]:
pos = POS["peak"]                         # (MATLAB row, column), the code's position
rows = []
for pos_label, p in [("code's position", POS["peak"]), ("centroid (variant)", POS["centroid"])]:
    for radius in (45, 50):
        n_used = 0
        for r in range(1, round((608 - 1) / 5) + 1):
            for c in range(1, round((700 - 140) / 5) + 1):
                coord = (r * 5 + 1, c * 5 + 140)
                if (np.hypot(p[:, 0] - coord[0], p[:, 1] - coord[1]) <= radius).sum() >= 32:
                    n_used += 1
        rows.append({"positions": pos_label, "radius_px": radius, "points_used": n_used})
grid = pd.DataFrame(rows)
print(grid.to_string(index=False))
rng_m = np.random.default_rng(691_23)
t0 = time.time()
for _ in range(4):
    cols = rng_m.choice(X_run.shape[1], 32, replace=False)
    ks.decoding_matrix(X_run[:, cols], LABELS, 8, ks.LAMBDA_OTHER, 1, rng_m)
sec32 = (time.time() - t0) / 4
print(f"one fit on 32 footprints (exact solver, as MATLAB's default for 100 or fewer predictors): {sec32:.2f} s")
for _, g in grid.iterrows():
    fits = g.points_used * 40 + 1000 * 40
    print(f"{g.positions}, {g.radius_px} px: {g.points_used} points x 40 fits + 1000 null decoders x 40 fits = {fits} fits, "
          f"{fits * sec32 / 3600:.1f} hours on one core, {fits * sec32 / 4 / 3600:.1f} on four")

**Result: blocked (not_runnable).** It depends on ed9b-octant-decoding, which is blocked. With the
code's positions, 5411 grid points are used at 45 pixels and 5905 at 50 pixels (5409 and 5897 with
centroids); with 40 fits per point and per null decoder that is 256,360 to 276,200 fits, about 10.2
to 11.0 hours on four cores.

In [ ]:
print(f"total run time {(time.time() - started) / 60:.1f} min")

## Summary

| Claim | Outcome (this attempt) | Outcome of -01 |
|---|---|---|
| ed9b-octant-decoding | blocked: not runnable | reproduced |
| ed9c-local-decoding-map | blocked: not runnable | reproduced |

How the analysis differs from the paper's. All 2948 footprints are used where the claims name
the 2,382 curated ones, which the dandiset does not mark; position is tracked from the behaviour
video with the code's rule; MATLAB's linear SVMs are replaced by scikit-learn's SGD (10 passes, for
more than 100 predictors) and LinearSVC (exact, for 100 or fewer), as MATLAB's defaults choose. Only
null reading (n1) is run; (n2) and the analyses that depend on these claims are costed, not run.
The first attempt (-01) judged ed9b and ed9c against decoders with shifted labels; the claims file now uses the paper's null, shuffled activity, with two readings of a null decoder.